# Init

In [2]:
import pandas as pd
import os
import pypdfium2 as pdfium
from pprint import pprint
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
import torch
import json
from deepdiff import DeepDiff
import xgrammar as xgr
import outlines

In [3]:
file_path = '../benchmark_truth/synthetic_tables/separate_files/aktiva_table_4_columns_span_False_thin_True_year_as_date_unit_in_first_cell_False_EUR_enumeration_True_0'
df = pd.read_csv(file_path+'.csv')

In [4]:
pdf = pdfium.PdfDocument(file_path+'.pdf')
text = pdf.get_page(0).get_textpage().get_text_range()#.replace('\r\n', ' ')
pprint(text)

('Aktiva 31.12.2013 31.12.2013 31.12.2012\r\n'
 'EUR EUR EUR\r\n'
 'A. Anlagevermögen\r\n'
 'I. Immaterielle Vermögensgegenstände\r\n'
 '1. geleistete Anzahlungen 913.637,50 5.560.454,30\r\n'
 '2. entgeltlich erworbene Konzessionen, gewerbliche Schutzrechte und ähnliche '
 'Rechte und Werte sowie Lizenzen an\r\n'
 'solchen Rechten und Werten\r\n'
 '9.377.116,62 2.855.838,45\r\n'
 '10.290.754,12 8.416.292,75\r\n'
 'II. Sachanlagen\r\n'
 '1. Grundstücke, grundstücksgleiche Rechte und Bauten einschließlich der '
 'Bauten auf fremden Grundstücken 1.954.774,63 7.016.617,35\r\n'
 '2. Technische Anlagen und Maschinen 254.563,64 5.589.517,35\r\n'
 '3. Andere Anlagen, Betriebs- und Geschäftsausstattung 8.772.123,75 '
 '4.788.619,96\r\n'
 '10.981.462,02 17.394.754,66\r\n'
 'III. Finanzanlagen\r\n'
 '1. Sonstige Finanzanlagen 1.689.839,31 7.853.816,31\r\n'
 '2. Ausleihungen an verbundene Unternehmen 8.880.276,96 5.725.104,93\r\n'
 '3. Ausleihungen an Unternehmen, mit denen ein Beteiligungsverhält

/usr/local/lib/python3.12/dist-packages/pypdfium2/_helpers/textpage.py:80: UserWarning: get_text_range() call with default params will be implicitly redirected to get_text_bounded()
  warnings.warn("get_text_range() call with default params will be implicitly redirected to get_text_bounded()")


In [5]:
markdown_table = """
| Aktiva                                                                                                                                                                                        | 31.12.2013   | 31.12.2013    | 31.12.2012    |
|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|--------------|---------------|---------------|
|                                                                                                                                                                                               | EUR          | EUR           | EUR           |
| A.<br>Anlagevermögen                                                                                                                                                                          |              |               |               |
| I.<br>Immaterielle<br>Vermögensgegenstände                                                                                                                                                    |              |               |               |
| 1.<br>geleistete<br>Anzahlungen                                                                                                                                                               | 913.637,50   |               | 5.560.454,30  |
| 2.<br>entgeltlich<br>erworbene<br>Konzessionen,<br>gewerbliche<br>Schutzrechte<br>und<br>ähnliche<br>Rechte<br>und<br>Werte<br>sowie<br>Lizenzen<br>an<br>solchen<br>Rechten<br>und<br>Werten | 9.377.116,62 |               | 2.855.838,45  |
|                                                                                                                                                                                               |              | 10.290.754,12 | 8.416.292,75  |
| II.<br>Sachanlagen                                                                                                                                                                            |              |               |               |
| 1.<br>Grundstücke,<br>grundstücksgleiche<br>Rechte<br>und<br>Bauten<br>einschließlich<br>der<br>Bauten<br>auf fremden<br>Grundstücken                                                         | 1.954.774,63 |               | 7.016.617,35  |
| 2.<br>Technische<br>Anlagen<br>und<br>Maschinen                                                                                                                                               | 254.563,64   |               | 5.589.517,35  |
| 3.<br>Andere<br>Anlagen,<br>Betriebs-<br>und<br>Geschäftsausstattung                                                                                                                          | 8.772.123,75 |               | 4.788.619,96  |
|                                                                                                                                                                                               |              | 10.981.462,02 | 17.394.754,66 |
| III.<br>Finanzanlagen                                                                                                                                                                         |              |               |               |
| 1.<br>Sonstige<br>Finanzanlagen                                                                                                                                                               | 1.689.839,31 |               | 7.853.816,31  |
| 2.<br>Ausleihungen<br>an<br>verbundene<br>Unternehmen                                                                                                                                         | 8.880.276,96 |               | 5.725.104,93  |
| 3.<br>Ausleihungen<br>an<br>Unternehmen,<br>mit<br>denen<br>ein<br>Beteiligungsverhältnis<br>besteht                                                                                          | 1.513.993,65 |               | 2.979.462,34  |
| 4.<br>Wertpapiere<br>des<br>Anlagevermögens                                                                                                                                                   | 5.072.841,30 |               | 4.370.153,18  |
| 5.<br>Sonstige<br>Ausleihungen                                                                                                                                                                | 5.901.836,02 |               | 371.321,42    |
|                                                                                                                                                                                               |              | 23.058.787,24 | 21.299.858,17 |
|                                                                                                                                                                                               |              | 44.331.003,39 | 47.110.905,58 |
| B.<br>Umlaufvermögen                                                                                                                                                                          |              |               |               |
| I.<br>Vorräte                                                                                                                                                                                 |              |               |               |
| 1.<br>Roh-,<br>Hilfs-<br>und<br>Betriebsstoffe                                                                                                                                                | 8.109.998,25 |               | 308.533,41    |
| 2.<br>Unfertige<br>Erzeugnisse,<br>unfertige<br>Leistungen                                                                                                                                    | 8.307.370,71 |               | 5.927.641,65  |
| 3.<br>Geleistete<br>Anzahlungen                                                                                                                                                               | 7.198.003,36 |               | 7.837.360,04  |
|                                                                                                                                                                                               |              | 23.615.372,32 | 14.073.535,09 |
| II.<br>Forderungen<br>und<br>sonstige<br>Vermögensgegenstände                                                                                                                                 |              |               |               |
| 1.<br>Forderungen<br>gegen<br>verbundene<br>Unternehmen                                                                                                                                       | 7.677.530,11 |               | 7.064.122,69  |
| 2.<br>Sonstige<br>Vermögensgegenstände                                                                                                                                                        | 160.325,22   |               | 3.778.111,64  |
|                                                                                                                                                                                               |              | 7.837.855,33  | 10.842.234,33 |
| III.<br>Wertpapiere                                                                                                                                                                           |              |               |               |
| 1.<br>Sonstige<br>Wertpapiere                                                                                                                                                                 |              | 3.813.647,37  | 8.195.312,06  |
| IV.<br>Kassenbestand,<br>Bundesbankguthaben,<br>Guthaben<br>bei Kreditinstituten<br>und<br>Schecks                                                                                            |              | 6.258.615,71  | 7.443.915,18  |
|                                                                                                                                                                                               |              | 41.525.490,73 | 40.554.996,67 |
| C.<br>Aktiver<br>Unterschiedsbetrag<br>aus<br>der<br>Vermögensverrechnung                                                                                                                     |              | 1.755.614,53  | 89.454,30     |
|                                                                                                                                                                                               |              | 87.612.108,66 | 87.755.356,55 |
|                                                                                                                                                                                               |              |               |               |
"""

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
import torch

device_map = "auto"
model_name = "Qwen/Qwen2.5-7B-Instruct"
# model_name = "Qwen/Qwen2.5-0.5B-Instruct"
# model_name = "meta-llama/Llama-3.1-8B-Instruct"
# model_name = "microsoft/Phi-4-mini-instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float32, device_map=device_map
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
config = AutoConfig.from_pretrained(model_name)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
[2025-06-17 08:45:18] INFO modeling.py:991: We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [16]:
def solve_task(prompts, table, max_tokens=2048, json_grammar=False, ebnf_str=""):
    if isinstance(prompts, list):
        messages = [
            {"role": "system", "content": "You are a helpful assistant that extracts information from tables."}
        ]
        for prompt in prompts:
            messages.append({"role": "user", "content": prompt})
        messages.append({
            "role": "user",
            "content": (
                "This is the table:\n"
                "```\n"
                f"{table}\n"
                "```\n"
            ),
        })
    else:
        messages = [
            {
                "role": "system",
                "content": "You are a helpful assistant that extracts information from tables.",
            },
            {
                "role": "user",
                "content": prompts,
            },
            {
                "role": "user",
                "content": (
                    "This is the table:\n"
                    "```\n"
                    f"{table}\n"
                    "```\n"
                ),
            },
        ]

    pprint(messages)

    texts = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer(texts, return_tensors="pt").to(model.device)

    if json_grammar:
        tokenizer_info = xgr.TokenizerInfo.from_huggingface(tokenizer, vocab_size=config.vocab_size)
        grammar_compiler = xgr.GrammarCompiler(tokenizer_info)
        compiled_grammar = grammar_compiler.compile_builtin_json_grammar()

        xgr_logits_processor = xgr.contrib.hf.LogitsProcessor(compiled_grammar)
        generated_ids = model.generate(
            **model_inputs, max_new_tokens=max_tokens, logits_processor=[xgr_logits_processor]
        )
        generated_ids = generated_ids[0][len(model_inputs.input_ids[0]) :]
        result = tokenizer.decode(generated_ids, skip_special_tokens=True)
    elif not ebnf_str == "":
        tokenizer_info = xgr.TokenizerInfo.from_huggingface(tokenizer, vocab_size=config.vocab_size)
        grammar_compiler = xgr.GrammarCompiler(tokenizer_info)
        # Grammar string that represents a JSON schema
        compiled_grammar = grammar_compiler.compile_grammar(ebnf_str)

        xgr_logits_processor = xgr.contrib.hf.LogitsProcessor(compiled_grammar)
        generated_ids = model.generate(
            **model_inputs, max_new_tokens=max_tokens, logits_processor=[xgr_logits_processor]
        )
        generated_ids = generated_ids[0][len(model_inputs.input_ids[0]) :]
        result = tokenizer.decode(generated_ids, skip_special_tokens=True)
    else:
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_tokens,
        )

        answer_start = model_inputs['input_ids'].shape[1]
        result = tokenizer.decode(generated_ids[0][answer_start:], skip_special_tokens=True)
    return result

In [8]:
file_path2 = '../benchmark_truth/synthetic_tables/separate_files/aktiva_table_4_columns_span_False_thin_True_year_as_date_unit_in_first_cell_False_€_enumeration_True_0'
df2 = pd.read_csv(file_path2+'.csv')

pdf2 = pdfium.PdfDocument(file_path2+'.pdf')
text2 = pdf2.get_page(0).get_textpage().get_text_range()
print(text2)

Aktiva 31.12.2020 31.12.2020 31.12.2019
€ € €
A. Anlagevermögen
I. Immaterielle Vermögensgegenstände
1. Geschäfts- oder Firmenwert 1.492.324,17 4.042.835,13
2. entgeltlich erworbene Konzessionen, gewerbliche Schutzrechte und ähnliche Rechte und Werte sowie Lizenzen an
solchen Rechten und Werten
6.847.537,50 1.129.665,06
8.339.861,67 5.172.500,19
II. Sachanlagen
1. Technische Anlagen und Maschinen 9.257.856,98 2.839.214,91
2. Andere Anlagen, Betriebs- und Geschäftsausstattung 932.533,00 5.130.595,26
3. geleistete Anzahlungen und Anlagen im Bau 7.769.993,64 8.108.088,71
17.960.383,61 16.077.898,88
III. Finanzanlagen
1. Ausleihungen an Unternehmen, mit denen ein Beteiligungsverhältnis besteht 1.539.169,41 4.344.342,95
27.839.414,69 25.594.742,02
B. Umlaufvermögen
I. Vorräte
1. Geleistete Anzahlungen 1.770.435,81 1.052.034,11
II. Forderungen und sonstige Vermögensgegenstände
1. Sonstige Vermögensgegenstände 9.759.428,49 5.776.695,97
11.529.864,30 6.828.730,08
39.369.278,99 32.423.472,09


/usr/local/lib/python3.12/dist-packages/pypdfium2/_helpers/textpage.py:80: UserWarning: get_text_range() call with default params will be implicitly redirected to get_text_bounded()
  warnings.warn("get_text_range() call with default params will be implicitly redirected to get_text_bounded()")


In [9]:
df2.dropna(subset=df2.columns[-2:]).reset_index(drop=True)

,E1,E2,E3,31.12.2020,31.12.2019
0,Anlagevermögen,Immaterielle Vermögensgegenstände,Geschäfts- oder Firmenwert,1.492324e+06,4.042835e+06
1,Anlagevermögen,Immaterielle Vermögensgegenstände,"entgeltlich erworbene Konzessionen, gewerblich...",6.847537e+06,1.129665e+06
2,Anlagevermögen,Sachanlagen,Technische Anlagen und Maschinen,9.257857e+06,2.839215e+06
3,Anlagevermögen,Sachanlagen,"Andere Anlagen, Betriebs- und Geschäftsausstat...",9.325330e+05,5.130595e+06
4,Anlagevermögen,Sachanlagen,geleistete Anzahlungen und Anlagen im Bau,7.769994e+06,8.108089e+06
5,Anlagevermögen,Finanzanlagen,"Ausleihungen an Unternehmen, mit denen ein Bet...",1.539169e+06,4.344343e+06
6,Umlaufvermögen,Vorräte,Geleistete Anzahlungen,1.770436e+06,1.052034e+06
7,Umlaufvermögen,Forderungen und sonstige Vermögensgegenstände,Sonstige Vermögensgegenstände,9.759428e+06,5.776696e+06


In [10]:
import json

def parse_json(string):
    # Remove code block markers and extra whitespace, then parse as JSON
    json_str = string.strip()
    if json_str.startswith("```json"):
        json_str = json_str[len("```json"):].strip()
    if json_str.endswith("```"):
        json_str = json_str[:-3].strip()
    parsed_json = json.loads(json_str)
    return parsed_json

In [11]:
def generate_json(rows):
    json_data = []
    for row in rows:
        json_row = {
            'type': row[0],
            'year': row[1],
            'previous_year': row[2]
        }
        json_data.append(json_row)
    return json_data

def generate_row_list(df, add_enumeration=True):
    rows = []
    enum = [0] * 3
    cols = [col for col in df.columns if col not in ['E1', 'E2', 'E3']]

    for key in df['E1'].unique():
        enum[0] += 1
        enum[1] = 0
        enum[2] = 0
        df_temp = df[df['E1'] == key]
        title = (enumerators[0][enum[0]-1] + ' ' + key) if add_enumeration else key

        if df_temp.shape[0] == 1 and check_na_or_given_string(df_temp['E2'].iloc[0], 'SUMME') and check_na_or_given_string(df_temp['E3'].iloc[0], 'SUMME'):
            rows.append([title] + df_temp[cols].iloc[0].tolist())

        else:
            rows.append([title] + [''] * (len(df.columns) - 3))

            for sub_key in df_temp['E2'].unique():
                df_sub_temp = df_temp[df_temp['E2'] == sub_key]
                enum[1] += 1
                enum[2] = 0
                title = (enumerators[1][enum[1]-1] + ' ' + sub_key) if add_enumeration else sub_key

                if df_sub_temp.shape[0] == 1 and check_na_or_given_string(df_sub_temp['E3'].iloc[0], 'SUMME'):
                    rows.append([title] + df_sub_temp[cols].iloc[0].tolist())
                else:
                    rows.append([title] + [''] * (len(df.columns) - 3))

                    for item in df_sub_temp['E3'].unique():
                        df_item_temp = df_sub_temp[df_sub_temp['E3'] == item]
                        enum[2] += 1
                        title = (enumerators[2][enum[2]-1] + ' ' + item) if add_enumeration else item

                        # if df_item_temp.shape[0] == 1:
                        if df_sub_temp.shape[0] == 1:
                            rows.append([title] + df_item_temp[cols].iloc[0].tolist())
                        else:
                            rows.append([title] + df_item_temp[cols].iloc[0].tolist())
    return rows

def check_na_or_given_string(value, string):
    return pd.isna(value) or (value == string)

In [12]:
baseprompt = """
Extract the information from the table below as JSON list. Each row should be an entry with three keys. The keys names are "type", "year", "previous_year".

Skip the first one or two rows if they contain headers or units.

The values of interest for the "year" and "previous_year" keys are the values in the second and third or fourth and fifth columns, and of numeric type. They can be empty and be represented by ''.

The value for the field "type" should be strings. Drop rows where there is no value for "type".
"""

In [13]:
truth = generate_json(generate_row_list(df.dropna(subset=df.columns[-2:]).reset_index(drop=True), add_enumeration=False))

# Silly examples

In [9]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant that extracts information from tables.",
    },
    {
        "role": "user",
        "content": (
            "Extract the following information from the table:\n"
            "1. The total assets as of 2023.\n"
            "2. The total liabilities as of 2023.\n"
            "3. The equity as of 2023."
        ),
    },
    {
        "role": "user",
        "content": (
            "This is the table:\n"
            "```\n"
            f"{text}\n"
            "```\n"
        ),
    },
]

texts = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
model_inputs = tokenizer(texts, return_tensors="pt").to(model.device)

generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=1024,
        )

In [10]:
# Only print the model's answer, skipping the prompt and system/user messages
# Find the start of the assistant's answer using the input length
answer_start = model_inputs['input_ids'].shape[1]
result = tokenizer.decode(generated_ids[0][answer_start:], skip_special_tokens=True)
print(result)

It seems like there might be a misunderstanding because the provided table does not contain data for the year 2023. Instead, it provides financial data for the years 2013 and 2012. Here's the extracted information based on the available data:

1. **Total Assets as of 2013**: 87,612,108.66 EUR
2. **Total Liabilities as of 2013**: Not directly provided in the given table. We would need to subtract equity from total assets to find this, but equity is also not provided.
3. **Equity as of 2013**: Not directly provided in the given table.

If you have data for 2023 or if you need the 2013 data reformatted, please let me know!


# Tests

## From text

### to JSON

#### zero shot

In [ ]:
result = solve_task(baseprompt, table = text)

In [65]:
pprint(parse_json(result))


[{'previous_year': '2012', 'type': 'Anlagevermögen', 'year': '2013'},
 {'previous_year': '2012', 'type': 'geleistete Anzahlungen', 'year': '2013'},
 {'previous_year': '2012',
  'type': 'entgeltlich erworbene Konzessionen, gewerbliche Schutzrechte und '
          'ähnliche Rechte und Werte sowie Lizenzen an solchen Rechten und '
          'Werten',
  'year': '2013'},
 {'previous_year': '2012',
  'type': 'I. Immaterielle Vermögensgegenstände',
  'year': '2013'},
 {'previous_year': '2.855.838.45',
  'type': '1. geleistete Anzahlungen',
  'year': '5.560.454.30'},
 {'previous_year': '2012',
  'type': '9. entgeltlich erworbene Konzessionen, gewerbliche Schutzrechte und '
          'ähnliche Rechte und Werte sowie Lizenzen an solchen Rechten und '
          'Werten',
  'year': '2013'},
 {'previous_year': '8.416.292.75',
  'type': '2. entgeltlich erworbene Konzessionen, gewerbliche Schutzrechte und '
          'ähnliche Rechte und Werte sowie Lizenzen an solchen Rechten und '
          'Werten

In [66]:
json_result = parse_json(result)
# TODO: get gold standard json from table and compare

In [ ]:
diff = DeepDiff(truth, json_result, significant_digits=2, get_deep_distance=True)

In [68]:
pprint(diff)

{'deep_distance': 0.4099526066350711,
 'iterable_item_added': {'root[26]': {'previous_year': '7.064.122.69',
                                      'type': '1. Forderungen gegen verbundene '
                                              'Unternehmen',
                                      'year': '7.677.530.11'},
                         'root[27]': {'previous_year': '3.778.111.64',
                                      'type': '2. Sonstige '
                                              'Vermögensgegenstände',
                                      'year': '160.325.22'},
                         'root[28]': {'previous_year': '2012',
                                      'type': '7.837.855.33',
                                      'year': '2013'},
                         'root[29]': {'previous_year': '2012',
                                      'type': 'III. Wertpapiere',
                                      'year': '2013'},
                         'root[30]': {'previous_year': '8.1

In [69]:
diff.get_stats()

{'PASSES COUNT': 0,
 'DIFF COUNT': 217,
 'DISTANCE CACHE HIT COUNT': 0,
 'MAX PASS LIMIT REACHED': False,
 'MAX DIFF LIMIT REACHED': False}

In [70]:
diff2 = DeepDiff(truth, json_result, significant_digits=2, get_deep_distance=True, cutoff_intersection_for_pairs=1)
diff2.get_stats()

{'PASSES COUNT': 0,
 'DIFF COUNT': 217,
 'DISTANCE CACHE HIT COUNT': 0,
 'MAX PASS LIMIT REACHED': False,
 'MAX DIFF LIMIT REACHED': False}

In [18]:
mode = "zeroshot"
file_path = '../benchmark_truth/synthetic_tables/separate_files/'
csv_files = [f for f in os.listdir(file_path) if f.endswith('.csv')]

for csv_file in csv_files[0:5]:
    df = pd.read_csv(os.path.join(file_path, csv_file))
    truth = generate_json(generate_row_list(df.dropna(subset=df.columns[-2:]).reset_index(drop=True), add_enumeration=False))
    
    pdf = pdfium.PdfDocument(os.path.join(file_path, csv_file.replace('.csv', '.pdf')))
    text = pdf.get_page(0).get_textpage().get_text_bounded()#.replace('\r\n', ' ')

    if mode == "zeroshot":
        prompts = [
            baseprompt
        ]

    result = solve_task(prompts, table=text)
    json_result = parse_json(result)
    diff = DeepDiff(truth, json_result, significant_digits=2, get_deep_distance=True)
    print(diff ['deep_distance'])

0.2967581047381546
0.40878378378378377
0.40924092409240925
0.4166666666666667
0.3316708229426434


#### one shot

In [42]:
prompts = []

baseprompt = """
Extract the information from the table below as JSON list. Each row should be an entry with three keys. The keys names are "type", "year", "previous_year".

Skip the first one or two rows if they contain headers or units.

The values of interest for the "year" and "previous_year" keys are the values in the second and third or fourth and fifth columns, and of numeric type. They can be empty and be represented by "".

The value for the field "type" should be strings. Drop rows where there is no value for "type".
"""
prompts.append(baseprompt)

prompt = f"""
Here is an example of an input and the output you should produce:

Input:
{text2}
"""
prompts.append(prompt)

prompt ='''
Output:
[
{"type": "Anlagevermögen", "year": "", "previous_year": ""},
{"type": "Sachanlagen", "year": "", "previous_year": ""},
{"type": "Geschäfts- oder Firmenwert", "year": 1492324.17, "previous_year": 4042835.13},
{"type": "entgeltlich erworbene Konzessionen, gewerbliche Schutzrechte und ähnliche Rechte und Werte sowie Lizenzen an solchen Rechten und Werten", "year": 6847537.50, "previous_year": 1129665.06},
{"type": "Sachanlagen", "year": "", "previous_year": ""},
{"type": "Technische Anlagen und Maschinen", "year": 9257856.98, "previous_year": 2839214.91},
{"type": "Andere Anlagen, Betriebs- und Geschäftsausstattung", "year": 932533.00, "previous_year": 5130595.26},
{"type": "geleistete Anzahlungen und Anlagen im Bau", "year": 7769993.64, "previous_year": 8108088.71},
{"type": "Finanzanlagen", "year": "", "previous_year": ""},
{"type": "Ausleihungen an Unternehmen, mit denen ein Beteiligungsverhältnis besteht", "year": 1539169.41, "previous_year": 4344342.95},
{"type": "Umlaufvermögen", "year": "", "previous_year": ""},
{"type": "Vorräte", "year": "", "previous_year": ""},
{"type": "Geleistete Anzahlungen", "year": 1770435.81, "previous_year": 1052034.11},
{"type": "Forderungen und sonstige Vermögensgegenstände", "year": "", "previous_year": ""},
{"type": "Sonstige Vermögensgegenstände", "year": 9759428.49, "previous_year": 5776695.97}
]
'''
prompts.append(prompt)

result = solve_task(prompts, table = text)

In [43]:
json_result = parse_json(result)

In [47]:
truth

[{'type': 'Anlagevermögen', 'year': '', 'previous_year': ''},
 {'type': 'Immaterielle Vermögensgegenstände',
  'year': '',
  'previous_year': ''},
 {'type': 'Selbst geschaffene gewerbliche Schutzrechte und ähnliche Rechte und Werte',
  'year': 2045058.741908544,
  'previous_year': 263476.2039241745},
 {'type': 'Geschäfts- oder Firmenwert',
  'year': 3678916.685758353,
  'previous_year': 1647880.540110861},
 {'type': 'geleistete Anzahlungen',
  'year': 6167008.379223865,
  'previous_year': 3121960.7318933085},
 {'type': 'entgeltlich erworbene Konzessionen, gewerbliche Schutzrechte und ähnliche Rechte und Werte sowie Lizenzen an solchen Rechten und Werten',
  'year': 9792256.58726621,
  'previous_year': 291548.820734644},
 {'type': 'Sachanlagen', 'year': '', 'previous_year': ''},
 {'type': 'Grundstücke, grundstücksgleiche Rechte und Bauten einschließlich der Bauten auf fremden Grundstücken',
  'year': 6754238.154774453,
  'previous_year': 7312236.356308069},
 {'type': 'Technische Anlagen

In [44]:
pprint(result)

('```json\n'
 '[\n'
 '    {"type": "Immaterielle Vermögensgegenstände", "year": 2045.06, '
 '"previous_year": 263.48},\n'
 '    {"type": "Geschäfts- oder Firmenwert", "year": 3678.92, "previous_year": '
 '1647.88},\n'
 '    {"type": "geleistete Anzahlungen", "year": 6167.01, "previous_year": '
 '3121.96},\n'
 '    {"type": "entgeltlich erworbene Konzessionen, gewerbliche Schutzrechte '
 'und ähnliche Rechte und Werte sowie Lizenzen an solchen Rechten und Werten", '
 '"year": 9792.26, "previous_year": 291.55},\n'
 '    {"type": "geleistete Anzahlungen", "year": 21683.24, "previous_year": '
 '5324.87},\n'
 '    {"type": "Grundstücke, grundstücksgleiche Rechte und Bauten '
 'einschließlich der Bauten auf fremden Grundstücken", "year": 6754.24, '
 '"previous_year": 7312.24},\n'
 '    {"type": "Technische Anlagen und Maschinen", "year": 3903.71, '
 '"previous_year": 2715.5},\n'
 '    {"type": "Andere Anlagen, Betriebs- und Geschäftsausstattung", "year": '
 '8508.95, "previous_year": 7829.68

In [50]:
from deepdiff import DeepDiff

diff = DeepDiff(truth, json_result, significant_digits=2, get_deep_distance=True)

In [51]:
pprint(diff)

{'deep_distance': 0.29540481400437635,
 'iterable_item_removed': {'root[29]': {'previous_year': '',
                                        'type': 'Wertpapiere',
                                        'year': ''},
                           'root[30]': {'previous_year': 9343555.56033837,
                                        'type': 'Anteile an verbundenen '
                                                'Unternehmen',
                                        'year': 8632129.5493096},
                           'root[31]': {'previous_year': 8480322.66549547,
                                        'type': 'Sonstige Wertpapiere',
                                        'year': 9121827.337552425},
                           'root[32]': {'previous_year': 6372152.819137012,
                                        'type': 'Kassenbestand, '
                                                'Bundesbankguthaben, Guthaben '
                                                'bei Kreditinstituten

In [63]:
DeepDiff(
    {"hey": 1}, 
    {"hey": 2, "ho": 'a'}, 
    get_deep_distance=True
)

{'dictionary_item_added': ["root['ho']"],
 'values_changed': {"root['hey']": {'new_value': 2, 'old_value': 1}},
 'deep_distance': 0.25}

In [19]:
mode = "oneshot"
file_path = '../benchmark_truth/synthetic_tables/separate_files/'
csv_files = [f for f in os.listdir(file_path) if f.endswith('.csv')]

for csv_file in csv_files[0:5]:
    df = pd.read_csv(os.path.join(file_path, csv_file))
    truth = generate_json(generate_row_list(df.dropna(subset=df.columns[-2:]).reset_index(drop=True), add_enumeration=False))
    
    pdf = pdfium.PdfDocument(os.path.join(file_path, csv_file.replace('.csv', '.pdf')))
    text = pdf.get_page(0).get_textpage().get_text_bounded()#.replace('\r\n', ' ')

    match mode:
        case "zeroshot":
            prompts = [
                baseprompt
            ]
        case "oneshot":
            prompts = []
            prompts.append(baseprompt)

            prompt = f"""
            Here is an example of an input and the output you should produce:

            Input:
            {text2}
            """
            prompts.append(prompt)

            prompt ="""
            Output:
            [
            {'type': 'Anlagevermögen', 'year': '', 'previous_year': ''},
            {'type': 'Sachanlagen', 'year': '', 'previous_year': ''},
            {'type': 'Geschäfts- oder Firmenwert', 'year': 1492324.17, 'previous_year': 4042835.13},
            {'type': 'entgeltlich erworbene Konzessionen, gewerbliche Schutzrechte und ähnliche Rechte und Werte sowie Lizenzen an solchen Rechten und Werten', 'year': 6847537.50, 'previous_year': 1129665.06},
            {'type': 'Sachanlagen', 'year': '', 'previous_year': ''},
            {'type': 'Technische Anlagen und Maschinen', 'year': 9257856.98, 'previous_year': 2839214.91},
            {'type': 'Andere Anlagen, Betriebs- und Geschäftsausstattung', 'year': 932533.00, 'previous_year': 5130595.26},
            {'type': 'geleistete Anzahlungen und Anlagen im Bau', 'year': 7769993.64, 'previous_year': 8108088.71},
            {'type': 'Finanzanlagen', 'year': '', 'previous_year': ''},
            {'type': 'Ausleihungen an Unternehmen, mit denen ein Beteiligungsverhältnis besteht', 'year': 1539169.41, 'previous_year': 4344342.95},
            {'type': 'Umlaufvermögen', 'year': '', 'previous_year': ''},
            {'type': 'Vorräte', 'year': '', 'previous_year': ''},
            {'type': 'Geleistete Anzahlungen', 'year': 1770435.81, 'previous_year': 1052034.11},
            {'type': 'Forderungen und sonstige Vermögensgegenstände', 'year': '', 'previous_year': ''},
            {'type': 'Sonstige Vermögensgegenstände', 'year': 9759428.49, 'previous_year': 5776695.97}
            ]
            """
            prompts.append(prompt)

    result = solve_task(prompts, table=text)
    json_result = parse_json(result)
    diff = DeepDiff(truth, json_result, significant_digits=2, get_deep_distance=True)
    print(diff ['deep_distance'])

0.30196936542669583
0.2618556701030928
0.31891891891891894
0.3045045045045045
0.2791970802919708


In [20]:
truth2 = generate_json(generate_row_list(df2.dropna(subset=df2.columns[-2:]).reset_index(drop=True), add_enumeration=False))
truth2

[{'type': 'Anlagevermögen', 'year': '', 'previous_year': ''},
 {'type': 'Immaterielle Vermögensgegenstände',
  'year': '',
  'previous_year': ''},
 {'type': 'Geschäfts- oder Firmenwert',
  'year': 1492324.1707022183,
  'previous_year': 4042835.1268538297},
 {'type': 'entgeltlich erworbene Konzessionen, gewerbliche Schutzrechte und ähnliche Rechte und Werte sowie Lizenzen an solchen Rechten und Werten',
  'year': 6847537.497453037,
  'previous_year': 1129665.05859581},
 {'type': 'Sachanlagen', 'year': '', 'previous_year': ''},
 {'type': 'Technische Anlagen und Maschinen',
  'year': 9257856.97692014,
  'previous_year': 2839214.911641954},
 {'type': 'Andere Anlagen, Betriebs- und Geschäftsausstattung',
  'year': 932532.9973218456,
  'previous_year': 5130595.262512061},
 {'type': 'geleistete Anzahlungen und Anlagen im Bau',
  'year': 7769993.638215843,
  'previous_year': 8108088.708784093},
 {'type': 'Finanzanlagen', 'year': '', 'previous_year': ''},
 {'type': 'Ausleihungen an Unternehmen,

In [23]:
print(json.dumps(truth2, indent=4))

[
    {
        "type": "Anlageverm\u00f6gen",
        "year": "",
        "previous_year": ""
    },
    {
        "type": "Immaterielle Verm\u00f6gensgegenst\u00e4nde",
        "year": "",
        "previous_year": ""
    },
    {
        "type": "Gesch\u00e4fts- oder Firmenwert",
        "year": 1492324.1707022183,
        "previous_year": 4042835.1268538297
    },
    {
        "type": "entgeltlich erworbene Konzessionen, gewerbliche Schutzrechte und \u00e4hnliche Rechte und Werte sowie Lizenzen an solchen Rechten und Werten",
        "year": 6847537.497453037,
        "previous_year": 1129665.05859581
    },
    {
        "type": "Sachanlagen",
        "year": "",
        "previous_year": ""
    },
    {
        "type": "Technische Anlagen und Maschinen",
        "year": 9257856.97692014,
        "previous_year": 2839214.911641954
    },
    {
        "type": "Andere Anlagen, Betriebs- und Gesch\u00e4ftsausstattung",
        "year": 932532.9973218456,
        "previous_year": 51305

In [24]:
mode = "oneshot"
file_path = '../benchmark_truth/synthetic_tables/separate_files/'
csv_files = [f for f in os.listdir(file_path) if f.endswith('.csv')]

for csv_file in csv_files[0:5]:
    df = pd.read_csv(os.path.join(file_path, csv_file))
    truth = generate_json(generate_row_list(df.dropna(subset=df.columns[-2:]).reset_index(drop=True), add_enumeration=False))
    
    pdf = pdfium.PdfDocument(os.path.join(file_path, csv_file.replace('.csv', '.pdf')))
    text = pdf.get_page(0).get_textpage().get_text_bounded()#.replace('\r\n', ' ')

    match mode:
        case "zeroshot":
            prompts = [
                baseprompt
            ]
        case "oneshot":
            prompts = []
            prompts.append(baseprompt)

            prompt = f"""
            Here is an example of an input and the output you should produce:

            Input:
            {text2}
            """
            prompts.append(prompt)

            prompt = f"""
            Output:
            {truth2}
            """
            prompts.append(prompt)

    result = solve_task(prompts, table=text)
    json_result = parse_json(result)
    diff = DeepDiff(truth, json_result, significant_digits=2, get_deep_distance=True)
    print(diff ['deep_distance'])

0.3201320132013201
0.1192468619246862
0.116600790513834
0.30699774266365687
0.31189083820662766


In [52]:
import numpy as np

def compare_dict_lists(list1, list2, keys=None, atol=1e-2):
    """
    Compare two lists of dictionaries.
    - Checks length, order, and values for each key.
    - If keys is provided, only compare those keys.
    - Numeric values are compared with absolute tolerance atol.
    Returns a dict with comparison results.
    """

    result = {
        "length_equal": len(list1) == len(list2),
        "length_1": len(list1),
        "length_2": len(list2),
        "order_equal": False,
        "mismatches": [],
        "missing_in_1": [],
        "missing_in_2": [],
    }

    # If keys not provided, use all keys from both lists
    if keys is None and list1 and list2:
        keys = set(list1[0].keys()) | set(list2[0].keys())
    elif keys is None:
        keys = set()

    # Check order and values
    order_equal = True
    mismatches = []
    min_len = min(len(list1), len(list2))
    for i in range(min_len):
        d1 = list1[i]
        d2 = list2[i]
        for k in keys:
            v1 = d1.get(k, None)
            v2 = d2.get(k, None)
            if isinstance(v1, (int, float)) and isinstance(v2, (int, float)):
                if not np.isclose(v1, v2, atol=atol, equal_nan=True):
                    mismatches.append({"index": i, "key": k, "v1": v1, "v2": v2})
                    order_equal = False
            else:
                if v1 != v2:
                    mismatches.append({"index": i, "key": k, "v1": v1, "v2": v2})
                    order_equal = False
    result["order_equal"] = order_equal
    result["mismatches"] = mismatches

    # Check for missing dicts (by 'type' key if present, else by all keys)
    def dict_id(d):
        if "type" in d:
            return d["type"]
        return tuple((k, d.get(k, None)) for k in sorted(keys))

    ids1 = set(dict_id(d) for d in list1)
    ids2 = set(dict_id(d) for d in list2)
    result["missing_in_1"] = [d for d in list2 if dict_id(d) not in ids1]
    result["missing_in_2"] = [d for d in list1 if dict_id(d) not in ids2]

    return result

compare_dict_lists(
    truth, json_result, keys=["type", "year", "previous_year"], atol=1e-2
)

{'length_equal': False,
 'length_1': 36,
 'length_2': 29,
 'order_equal': False,
 'mismatches': [{'index': 0,
   'key': 'type',
   'v1': 'Anlagevermögen',
   'v2': 'Immaterielle Vermögensgegenstände'},
  {'index': 0, 'key': 'year', 'v1': '', 'v2': 2045.06},
  {'index': 0, 'key': 'previous_year', 'v1': '', 'v2': 263.48},
  {'index': 1,
   'key': 'type',
   'v1': 'Immaterielle Vermögensgegenstände',
   'v2': 'Geschäfts- oder Firmenwert'},
  {'index': 1, 'key': 'year', 'v1': '', 'v2': 3678.92},
  {'index': 1, 'key': 'previous_year', 'v1': '', 'v2': 1647.88},
  {'index': 2,
   'key': 'type',
   'v1': 'Selbst geschaffene gewerbliche Schutzrechte und ähnliche Rechte und Werte',
   'v2': 'geleistete Anzahlungen'},
  {'index': 2, 'key': 'year', 'v1': 2045058.741908544, 'v2': 6167.01},
  {'index': 2, 'key': 'previous_year', 'v1': 263476.2039241745, 'v2': 3121.96},
  {'index': 3,
   'key': 'type',
   'v1': 'Geschäfts- oder Firmenwert',
   'v2': 'entgeltlich erworbene Konzessionen, gewerbliche Sc

### to JSON restricted

In [ ]:
mode = "oneshot"
file_path = '../benchmark_truth/synthetic_tables/separate_files/'
csv_files = [f for f in os.listdir(file_path) if f.endswith('.csv')]

for csv_file in csv_files[0:5]:
    df = pd.read_csv(os.path.join(file_path, csv_file))
    truth = generate_json(generate_row_list(df.dropna(subset=df.columns[-2:]).reset_index(drop=True), add_enumeration=False))
    
    pdf = pdfium.PdfDocument(os.path.join(file_path, csv_file.replace('.csv', '.pdf')))
    text = pdf.get_page(0).get_textpage().get_text_bounded()#.replace('\r\n', ' ')

    match mode:
        case "zeroshot":
            prompts = [
                baseprompt
            ]
        case "oneshot":
            prompts = []
            prompts.append(baseprompt)

            prompt = f"""
            Here is an example of an input and the output you should produce:

            Input:
            {text2}
            """
            prompts.append(prompt)

            prompt ="""
            Output:
            [
            {'type': 'Anlagevermögen', 'year': '', 'previous_year': ''},
            {'type': 'Sachanlagen', 'year': '', 'previous_year': ''},
            {'type': 'Geschäfts- oder Firmenwert', 'year': 1492324.17, 'previous_year': 4042835.13},
            {'type': 'entgeltlich erworbene Konzessionen, gewerbliche Schutzrechte und ähnliche Rechte und Werte sowie Lizenzen an solchen Rechten und Werten', 'year': 6847537.50, 'previous_year': 1129665.06},
            {'type': 'Sachanlagen', 'year': '', 'previous_year': ''},
            {'type': 'Technische Anlagen und Maschinen', 'year': 9257856.98, 'previous_year': 2839214.91},
            {'type': 'Andere Anlagen, Betriebs- und Geschäftsausstattung', 'year': 932533.00, 'previous_year': 5130595.26},
            {'type': 'geleistete Anzahlungen und Anlagen im Bau', 'year': 7769993.64, 'previous_year': 8108088.71},
            {'type': 'Finanzanlagen', 'year': '', 'previous_year': ''},
            {'type': 'Ausleihungen an Unternehmen, mit denen ein Beteiligungsverhältnis besteht', 'year': 1539169.41, 'previous_year': 4344342.95},
            {'type': 'Umlaufvermögen', 'year': '', 'previous_year': ''},
            {'type': 'Vorräte', 'year': '', 'previous_year': ''},
            {'type': 'Geleistete Anzahlungen', 'year': 1770435.81, 'previous_year': 1052034.11},
            {'type': 'Forderungen und sonstige Vermögensgegenstände', 'year': '', 'previous_year': ''},
            {'type': 'Sonstige Vermögensgegenstände', 'year': 9759428.49, 'previous_year': 5776695.97}
            ]
            """
            prompts.append(prompt)

    result = solve_task(prompts, table=text, json_grammar=True)
    json_result = parse_json(result)
    diff = DeepDiff(truth, json_result, significant_digits=2, get_deep_distance=True)
    print(diff ['deep_distance'])

0.27522935779816515
0.1087866108786611
0.31891891891891894
0.2777777777777778
0.25693160813308685


#### full list (fill NA)

In [14]:
df_ebnf = df.copy()
df_ebnf = df_ebnf.drop(df_ebnf.columns[-2:], axis=1).reset_index(drop=True)
df_ebnf['year'] = "number_or_null"
df_ebnf['previous_year'] = "number_or_null"
print('root ::= "'+df_ebnf.to_json(orient='records', indent=0, force_ascii=False).replace('"', '\\"').replace('\\"number_or_null\\"', '" number_or_null "')+'"')

root ::= "[{\"E1\":\"Anlagevermögen\",\"E2\":\"Immaterielle Vermögensgegenstände\",\"E3\":\"Selbst geschaffene gewerbliche Schutzrechte und ähnliche Rechte und Werte\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Anlagevermögen\",\"E2\":\"Immaterielle Vermögensgegenstände\",\"E3\":\"Geschäfts- oder Firmenwert\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Anlagevermögen\",\"E2\":\"Immaterielle Vermögensgegenstände\",\"E3\":\"geleistete Anzahlungen\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Anlagevermögen\",\"E2\":\"Immaterielle Vermögensgegenstände\",\"E3\":\"entgeltlich erworbene Konzessionen, gewerbliche Schutzrechte und ähnliche Rechte und Werte sowie Lizenzen an solchen Rechten und Werten\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Anlagevermögen\",\"E2\":\"Sachanlagen\",\"E3\":\"Grundstücke, grundstücksgleiche Rechte und Bauten einschließlich der Baute

In [15]:
import re

df2_rounded = df2.rename(columns={"31.12.2020": "year", "31.12.2019": "previous_year"})
for col in ["year", "previous_year"]:
    if col in df2_rounded.columns:
        df2_rounded[col] = df2_rounded[col].apply(lambda x: f"{x:.2f}" if pd.notnull(x) else x)
s = df2_rounded.to_json(orient='records', indent=0, force_ascii=False)#.replace("null", '"null"')
s_fixed = re.sub(r'("year":)"([0-9\.\-e]+)"', r'\1\2', s)
s_fixed = re.sub(r'("previous_year":)"([0-9\.\-e]+)"', r'\1\2', s_fixed)
s_fixed

'[{"E1":"Anlagevermögen","E2":"Immaterielle Vermögensgegenstände","E3":"Selbst geschaffene gewerbliche Schutzrechte und ähnliche Rechte und Werte","year":null,"previous_year":null},{"E1":"Anlagevermögen","E2":"Immaterielle Vermögensgegenstände","E3":"Geschäfts- oder Firmenwert","year":1492324.17,"previous_year":4042835.13},{"E1":"Anlagevermögen","E2":"Immaterielle Vermögensgegenstände","E3":"geleistete Anzahlungen","year":null,"previous_year":null},{"E1":"Anlagevermögen","E2":"Immaterielle Vermögensgegenstände","E3":"entgeltlich erworbene Konzessionen, gewerbliche Schutzrechte und ähnliche Rechte und Werte sowie Lizenzen an solchen Rechten und Werten","year":6847537.50,"previous_year":1129665.06},{"E1":"Anlagevermögen","E2":"Sachanlagen","E3":"Grundstücke, grundstücksgleiche Rechte und Bauten einschließlich der Bauten auf fremden Grundstücken","year":null,"previous_year":null},{"E1":"Anlagevermögen","E2":"Sachanlagen","E3":"Technische Anlagen und Maschinen","year":9257856.98,"previous_

In [17]:
mode = "oneshot"
file_path = '../benchmark_truth/synthetic_tables/separate_files/'
csv_files = [f for f in os.listdir(file_path) if f.endswith('.csv')]

# ebnf_str = r"""
# root ::= "[" row* "]"
# row ::= "{" key_value1, key_value2, key_value3 "}"
# key_value1 ::= "\"" "type" "\"" ":"  "\"" text "\""
# key_value2 ::= "\"" "year" "\"" ":" number
# key_value3 ::= "\"" "previous_year" "\"" ":" number
# text ::= [a-zA-Z0-9äöüßÄÖÜ ,.-]+
# number ::= [0-9]+ ("." [0-9]+)? | ""
# """

# ebnf_str = r"""
# root ::= "[" row* "]"
# row ::= "{" key_value1 "," key_value2 "," key_value3 "}"
# key_value1 ::= "\"" "type" "\"" ":"  "\"" text "\""
# key_value2 ::= "\"" "year" "\"" ":" number
# key_value3 ::= "\"" "previous_year" "\"" ":" number
# text ::= [a-zA-Z0-9äöüßÄÖÜ ,.-]+
# number ::= [0-9]+ ("." [0-9]+)? | ""
# """

baseprompt2 = """
Extract the information from the table below as JSON list. Each row should be an entry with five keys. The keys names are "E1", "E2", "E3", "year", "previous_year".

The entries for "E1", "E2" and "E3" are given by an EBNF. You just have to extract the numeric values for "year" and "previous_year" from the second and third or fourth and fifth columns, respectively.

If there are no corresponding numeric values for a given triple of "E1", "E2" and "E3" it should be represented by "null".

Skip the first one or two rows if they contain headers or units.

Do not alter the numeric values. Just extract the numeric values as they are. Ignore the currency symbol and the thousands separator.
"""

ebnf_str = r"""
root ::= "[{\"E1\":\"Anlagevermögen\",\"E2\":\"Immaterielle Vermögensgegenstände\",\"E3\":\"Selbst geschaffene gewerbliche Schutzrechte und ähnliche Rechte und Werte\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Anlagevermögen\",\"E2\":\"Immaterielle Vermögensgegenstände\",\"E3\":\"Geschäfts- oder Firmenwert\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Anlagevermögen\",\"E2\":\"Immaterielle Vermögensgegenstände\",\"E3\":\"geleistete Anzahlungen\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Anlagevermögen\",\"E2\":\"Immaterielle Vermögensgegenstände\",\"E3\":\"entgeltlich erworbene Konzessionen, gewerbliche Schutzrechte und ähnliche Rechte und Werte sowie Lizenzen an solchen Rechten und Werten\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Anlagevermögen\",\"E2\":\"Sachanlagen\",\"E3\":\"Grundstücke, grundstücksgleiche Rechte und Bauten einschließlich der Bauten auf fremden Grundstücken\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Anlagevermögen\",\"E2\":\"Sachanlagen\",\"E3\":\"Technische Anlagen und Maschinen\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Anlagevermögen\",\"E2\":\"Sachanlagen\",\"E3\":\"Andere Anlagen, Betriebs- und Geschäftsausstattung\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Anlagevermögen\",\"E2\":\"Sachanlagen\",\"E3\":\"geleistete Anzahlungen und Anlagen im Bau\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Anlagevermögen\",\"E2\":\"Finanzanlagen\",\"E3\":\"Sonstige Finanzanlagen\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Anlagevermögen\",\"E2\":\"Finanzanlagen\",\"E3\":\"Anteile an verbundenen Unternehmen\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Anlagevermögen\",\"E2\":\"Finanzanlagen\",\"E3\":\"Ausleihungen an verbundene Unternehmen\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Anlagevermögen\",\"E2\":\"Finanzanlagen\",\"E3\":\"Ausleihungen an Unternehmen, mit denen ein Beteiligungsverhältnis besteht\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Anlagevermögen\",\"E2\":\"Finanzanlagen\",\"E3\":\"Wertpapiere des Anlagevermögens\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Anlagevermögen\",\"E2\":\"Finanzanlagen\",\"E3\":\"Sonstige Ausleihungen\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Umlaufvermögen\",\"E2\":\"Vorräte\",\"E3\":\"Roh-, Hilfs- und Betriebsstoffe\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Umlaufvermögen\",\"E2\":\"Vorräte\",\"E3\":\"Unfertige Erzeugnisse, unfertige Leistungen\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Umlaufvermögen\",\"E2\":\"Vorräte\",\"E3\":\"Fertige Erzeugnisse und Waren\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Umlaufvermögen\",\"E2\":\"Vorräte\",\"E3\":\"Geleistete Anzahlungen\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Umlaufvermögen\",\"E2\":\"Forderungen und sonstige Vermögensgegenstände\",\"E3\":\"Forderungen aus Lieferungen und Leistungen\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Umlaufvermögen\",\"E2\":\"Forderungen und sonstige Vermögensgegenstände\",\"E3\":\"Forderungen gegen verbundene Unternehmen\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Umlaufvermögen\",\"E2\":\"Forderungen und sonstige Vermögensgegenstände\",\"E3\":\"Forderungen gegen Unternehmen, mit denen ein Beteiligungsverhältnis besteht\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Umlaufvermögen\",\"E2\":\"Forderungen und sonstige Vermögensgegenstände\",\"E3\":\"Sonstige Vermögensgegenstände\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Umlaufvermögen\",\"E2\":\"Wertpapiere\",\"E3\":\"Anteile an verbundenen Unternehmen\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Umlaufvermögen\",\"E2\":\"Wertpapiere\",\"E3\":\"Sonstige Wertpapiere\",\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Umlaufvermögen\",\"E2\":\"Kassenbestand, Bundesbankguthaben, Guthaben bei Kreditinstituten und Schecks\",\"E3\":null,\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Rechnungsabgrenzungsposten\",\"E2\":null,\"E3\":null,\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Aktive latente Steuern\",\"E2\":null,\"E3\":null,\"year\":" number_or_null ",\"previous_year\":" number_or_null "},{\"E1\":\"Aktiver Unterschiedsbetrag aus der Vermögensverrechnung\",\"E2\":null,\"E3\":null,\"year\":" number_or_null ",\"previous_year\":" number_or_null "}]"
number_or_null ::= number | "null"
number ::= [0-9]+ ("." [0-9]+)?
"""

for csv_file in csv_files[0:1]:
    df = pd.read_csv(os.path.join(file_path, csv_file))

    df_rounded = df.rename(columns={df.columns[-2]: "year", df.columns[-1]: "previous_year"})
    for col in ["year", "previous_year"]:
        if col in df_rounded.columns:
            df_rounded[col] = df_rounded[col].apply(lambda x: f"{x/1000000:.2f}" if pd.notnull(x) else x)
    s = df_rounded.to_json(orient='records', indent=0, force_ascii=False)#.replace("null", '"null"')
    s_fixed = re.sub(r'("year":)"([0-9\.\-e]+)"', r'\1\2', s)
    s_fixed = re.sub(r'("previous_year":)"([0-9\.\-e]+)"', r'\1\2', s_fixed)

    truth = parse_json(s_fixed)
    
    pdf = pdfium.PdfDocument(os.path.join(file_path, csv_file.replace('.csv', '.pdf')))
    text = pdf.get_page(0).get_textpage().get_text_bounded()#.replace('\r\n', ' ')

    match mode:
        case "zeroshot":
            prompts = [
                baseprompt2
            ]
        case "oneshot":
            prompts = []
            prompts.append(baseprompt2)

            prompt = f"""
            Here is an example of an input and the output you should produce:

            Input:
            {text2}
            """
            prompts.append(prompt)

            prompt ="""
            Output:
            [{"E1":"Anlagevermögen","E2":"Immaterielle Vermögensgegenstände","E3":"Selbst geschaffene gewerbliche Schutzrechte und ähnliche Rechte und Werte","year":"null","previous_year":"null"},{"E1":"Anlagevermögen","E2":"Immaterielle Vermögensgegenstände","E3":"Geschäfts- oder Firmenwert","year":1492324.17,"previous_year":4042835.13},{"E1":"Anlagevermögen","E2":"Immaterielle Vermögensgegenstände","E3":"geleistete Anzahlungen","year":"null","previous_year":"null"},{"E1":"Anlagevermögen","E2":"Immaterielle Vermögensgegenstände","E3":"entgeltlich erworbene Konzessionen, gewerbliche Schutzrechte und ähnliche Rechte und Werte sowie Lizenzen an solchen Rechten und Werten","year":6847537.50,"previous_year":1129665.06},{"E1":"Anlagevermögen","E2":"Sachanlagen","E3":"Grundstücke, grundstücksgleiche Rechte und Bauten einschließlich der Bauten auf fremden Grundstücken","year":"null","previous_year":"null"},{"E1":"Anlagevermögen","E2":"Sachanlagen","E3":"Technische Anlagen und Maschinen","year":9257856.98,"previous_year":2839214.91},{"E1":"Anlagevermögen","E2":"Sachanlagen","E3":"Andere Anlagen, Betriebs- und Geschäftsausstattung","year":932533.00,"previous_year":5130595.26},{"E1":"Anlagevermögen","E2":"Sachanlagen","E3":"geleistete Anzahlungen und Anlagen im Bau","year":7769993.64,"previous_year":8108088.71},{"E1":"Anlagevermögen","E2":"Finanzanlagen","E3":"Sonstige Finanzanlagen","year":"null","previous_year":"null"},{"E1":"Anlagevermögen","E2":"Finanzanlagen","E3":"Anteile an verbundenen Unternehmen","year":"null","previous_year":"null"},{"E1":"Anlagevermögen","E2":"Finanzanlagen","E3":"Ausleihungen an verbundene Unternehmen","year":"null","previous_year":"null"},{"E1":"Anlagevermögen","E2":"Finanzanlagen","E3":"Ausleihungen an Unternehmen, mit denen ein Beteiligungsverhältnis besteht","year":1539169.41,"previous_year":4344342.95},{"E1":"Anlagevermögen","E2":"Finanzanlagen","E3":"Wertpapiere des Anlagevermögens","year":"null","previous_year":"null"},{"E1":"Anlagevermögen","E2":"Finanzanlagen","E3":"Sonstige Ausleihungen","year":"null","previous_year":"null"},{"E1":"Umlaufvermögen","E2":"Vorräte","E3":"Roh-, Hilfs- und Betriebsstoffe","year":"null","previous_year":"null"},{"E1":"Umlaufvermögen","E2":"Vorräte","E3":"Unfertige Erzeugnisse, unfertige Leistungen","year":"null","previous_year":"null"},{"E1":"Umlaufvermögen","E2":"Vorräte","E3":"Fertige Erzeugnisse und Waren","year":"null","previous_year":"null"},{"E1":"Umlaufvermögen","E2":"Vorräte","E3":"Geleistete Anzahlungen","year":1770435.81,"previous_year":1052034.11},{"E1":"Umlaufvermögen","E2":"Forderungen und sonstige Vermögensgegenstände","E3":"Forderungen aus Lieferungen und Leistungen","year":"null","previous_year":"null"},{"E1":"Umlaufvermögen","E2":"Forderungen und sonstige Vermögensgegenstände","E3":"Forderungen gegen verbundene Unternehmen","year":"null","previous_year":"null"},{"E1":"Umlaufvermögen","E2":"Forderungen und sonstige Vermögensgegenstände","E3":"Forderungen gegen Unternehmen, mit denen ein Beteiligungsverhältnis besteht","year":"null","previous_year":"null"},{"E1":"Umlaufvermögen","E2":"Forderungen und sonstige Vermögensgegenstände","E3":"Sonstige Vermögensgegenstände","year":9759428.49,"previous_year":5776695.97},{"E1":"Umlaufvermögen","E2":"Wertpapiere","E3":"Anteile an verbundenen Unternehmen","year":"null","previous_year":"null"},{"E1":"Umlaufvermögen","E2":"Wertpapiere","E3":"Sonstige Wertpapiere","year":"null","previous_year":"null"},{"E1":"Umlaufvermögen","E2":"Kassenbestand, Bundesbankguthaben, Guthaben bei Kreditinstituten und Schecks","E3":"null","year":"null","previous_year":"null"},{"E1":"Rechnungsabgrenzungsposten","E2":"null","E3":"null","year":"null","previous_year":"null"},{"E1":"Aktive latente Steuern","E2":"null","E3":"null","year":"null","previous_year":"null"},{"E1":"Aktiver Unterschiedsbetrag aus der Vermögensverrechnung","E2":"null","E3":"null","year":"null","previous_year":"null"}]
            """
            prompts.append(prompt)

    result = solve_task(prompts, table=text, ebnf_str=ebnf_str).replace('๏', 'äf').replace('࿌', 'üc')
    print(result)
    json_result = parse_json(result)
    diff = DeepDiff(truth, json_result, significant_digits=2, get_deep_distance=True)
    print(diff ['deep_distance'])

[{'content': 'You are a helpful assistant that extracts information from '
             'tables.',
  'role': 'system'},
 {'content': '\n'
             'Extract the information from the table below as JSON list. Each '
             'row should be an entry with five keys. The keys names are "E1", '
             '"E2", "E3", "year", "previous_year".\n'
             '\n'
             'The entries for "E1", "E2" and "E3" are given by an EBNF. You '
             'just have to extract the numeric values for "year" and '
             '"previous_year" from the second and third or fourth and fifth '
             'columns, respectively.\n'
             '\n'
             'If there are no corresponding numeric values for a given triple '
             'of "E1", "E2" and "E3" it should be represented by "null".\n'
             '\n'
             'Skip the first one or two rows if they contain headers or '
             'units.\n'
             '\n'
             'Do not alter the numeric values. Just ext

In [106]:
diff

{'type_changes': {"root[7]['year']": {'old_type': NoneType,
   'new_type': float,
   'old_value': None,
   'new_value': 22.84},
  "root[7]['previous_year']": {'old_type': NoneType,
   'new_type': float,
   'old_value': None,
   'new_value': 16.19},
  "root[21]['year']": {'old_type': NoneType,
   'new_type': float,
   'old_value': None,
   'new_value': 12.39},
  "root[21]['previous_year']": {'old_type': NoneType,
   'new_type': float,
   'old_value': None,
   'new_value': 24.17}},
 'values_changed': {"root[4]['E3']": {'new_value': 'Grundst࿌ke, grundst࿌ksgleiche Rechte und Bauten einschließlich der Bauten auf fremden Grundst࿌ken',
   'old_value': 'Grundstücke, grundstücksgleiche Rechte und Bauten einschließlich der Bauten auf fremden Grundstücken'}},
 'deep_distance': 0.021035598705501618}

In [98]:
text

'Aktiva (in Mio. EUR) 31.12.2006 31.12.2005\r\nAnlagevermögen\r\nImmaterielle Vermögensgegenstände\r\nSelbst geschaffene gewerbliche Schutzrechte und ähnliche Rechte und Werte 4,33 5,21\r\ngeleistete Anzahlungen 5,21 3,87\r\nentgeltlich erworbene Konzessionen, gewerbliche Schutzrechte und ähnliche Rechte und Werte sowie Lizenzen an solchen Rechten und\r\nWerten\r\n2,04 9,18\r\n11,58 18,26\r\nSachanlagen Grundstücke, grundstücksgleiche Rechte und Bauten einschließlich der Bauten auf fremden Grundstücken 9,88 7,74\r\nTechnische Anlagen und Maschinen 9,55 1,35\r\nAndere Anlagen, Betriebs- und Geschäftsausstattung 3,41 7,10\r\n22,84 16,19\r\nFinanzanlagen\r\nSonstige Finanzanlagen 2,25 2,73\r\nAnteile an verbundenen Unternehmen 2,21 8,27\r\nAusleihungen an verbundene Unternehmen 1,06 4,02\r\nAusleihungen an Unternehmen, mit denen ein Beteiligungsverhältnis besteht 7,74 2,07\r\nWertpapiere des Anlagevermögens 8,47 6,07\r\n21,73 23,16\r\n56,15 57,61\r\nUmlaufvermögen\r\nVorräte Roh-, Hilfs- 

In [45]:
df_rounded = df.round(2).rename(columns={df.columns[-2]: "Geschaeftsjahr", df.columns[-1]: "Vorjahr"})
df_rounded.to_json(orient='records', indent=0, force_ascii=False)

'[{"E1":"Anlagevermögen","E2":"Immaterielle Vermögensgegenstände","E3":"Selbst geschaffene gewerbliche Schutzrechte und ähnliche Rechte und Werte","Geschaeftsjahr":4330503.2699999996,"Vorjahr":5208993.0899999999},{"E1":"Anlagevermögen","E2":"Immaterielle Vermögensgegenstände","E3":"Geschäfts- oder Firmenwert","Geschaeftsjahr":null,"Vorjahr":null},{"E1":"Anlagevermögen","E2":"Immaterielle Vermögensgegenstände","E3":"geleistete Anzahlungen","Geschaeftsjahr":5212856.6100000003,"Vorjahr":3870170.0099999998},{"E1":"Anlagevermögen","E2":"Immaterielle Vermögensgegenstände","E3":"entgeltlich erworbene Konzessionen, gewerbliche Schutzrechte und ähnliche Rechte und Werte sowie Lizenzen an solchen Rechten und Werten","Geschaeftsjahr":2041393.73,"Vorjahr":9178313.1999999993},{"E1":"Anlagevermögen","E2":"Sachanlagen","E3":"Grundstücke, grundstücksgleiche Rechte und Bauten einschließlich der Bauten auf fremden Grundstücken","Geschaeftsjahr":9878678.5099999998,"Vorjahr":7744922.2999999998},{"E1":"Anl

In [107]:
json_result

[{'E1': 'Anlagevermögen',
  'E2': 'Immaterielle Vermögensgegenstände',
  'E3': 'Selbst geschaffene gewerbliche Schutzrechte und ähnliche Rechte und Werte',
  'year': 4.33,
  'previous_year': 5.21},
 {'E1': 'Anlagevermögen',
  'E2': 'Immaterielle Vermögensgegenstände',
  'E3': 'Geschäfts- oder Firmenwert',
  'year': None,
  'previous_year': None},
 {'E1': 'Anlagevermögen',
  'E2': 'Immaterielle Vermögensgegenstände',
  'E3': 'geleistete Anzahlungen',
  'year': 5.21,
  'previous_year': 3.87},
 {'E1': 'Anlagevermögen',
  'E2': 'Immaterielle Vermögensgegenstände',
  'E3': 'entgeltlich erworbene Konzessionen, gewerbliche Schutzrechte und ähnliche Rechte und Werte sowie Lizenzen an solchen Rechten und Werten',
  'year': 2.04,
  'previous_year': 9.18},
 {'E1': 'Anlagevermögen',
  'E2': 'Sachanlagen',
  'E3': 'Grundst࿌ke, grundst࿌ksgleiche Rechte und Bauten einschließlich der Bauten auf fremden Grundst࿌ken',
  'year': 9.88,
  'previous_year': 7.74},
 {'E1': 'Anlagevermögen',
  'E2': 'Sachanlag

In [109]:
df_rounded


,E1,E2,E3,year,previous_year
0,Anlagevermögen,Immaterielle Vermögensgegenstände,Selbst geschaffene gewerbliche Schutzrechte un...,4.33,5.21
1,Anlagevermögen,Immaterielle Vermögensgegenstände,Geschäfts- oder Firmenwert,NaN,NaN
2,Anlagevermögen,Immaterielle Vermögensgegenstände,geleistete Anzahlungen,5.21,3.87
3,Anlagevermögen,Immaterielle Vermögensgegenstände,"entgeltlich erworbene Konzessionen, gewerblich...",2.04,9.18
4,Anlagevermögen,Sachanlagen,"Grundstücke, grundstücksgleiche Rechte und Bau...",9.88,7.74
5,Anlagevermögen,Sachanlagen,Technische Anlagen und Maschinen,9.55,1.35
6,Anlagevermögen,Sachanlagen,"Andere Anlagen, Betriebs- und Geschäftsausstat...",3.41,7.10
7,Anlagevermögen,Sachanlagen,geleistete Anzahlungen und Anlagen im Bau,NaN,NaN
8,Anlagevermögen,Finanzanlagen,Sonstige Finanzanlagen,2.25,2.73
9,Anlagevermögen,Finanzanlagen,Anteile an verbundenen Unternehmen,2.21,8.27


In [75]:
result

'[{"E1":"Anlagevermögen","E2":"Immaterielle Vermögensgegenstände","E3":"Selbst geschaffene gewerbliche Schutzrechte und ähnliche Rechte und Werte","Geschaeftsjahr":4.33,"Vorjahr":5.21},{"E1":"Anlagevermögen","E2":"Immaterielle Vermögensgegenstände","E3":"Gesch๏ts- oder Firmenwert","Geschaeftsjahr":5.21,"Vorjahr":3.87},{"E1":"Anlagevermögen","E2":"Immaterielle Vermögensgegenstände","E3":"geleistete Anzahlungen","Geschaeftsjahr":5.21,"Vorjahr":3.87},{"E1":"Anlagevermögen","E2":"Immaterielle Vermögensgegenstände","E3":"entgeltlich erworbene Konzessionen, gewerbliche Schutzrechte und ähnliche Rechte und Werte sowie Lizenzen an solchen Rechten und Werten","Geschaeftsjahr":2.04,"Vorjahr":9.18},{"E1":"Anlagevermögen","E2":"Sachanlagen","E3":"Grundst࿌ke, grundst࿌ksgleiche Rechte und Bauten einschließlich der Bauten auf fremden Grundst࿌ken","Geschaeftsjahr":9.88,"Vorjahr":7.74},{"E1":"Anlagevermögen","E2":"Sachanlagen","E3":"Technische Anlagen und Maschinen","Geschaeftsjahr":9.55,"Vorjahr":1.35

### with outlines

In [14]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

model = outlines.models.transformers(model_name, device="cuda")

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [16]:
# You must apply the chat template tokens to the prompt!
# See below for an example.
prompt = """
<|im_start|>system
You extract information from text.
<|im_end|>

<|im_start|>user
What food does the following text describe?

Text: I really really really want pizza.
<|im_end|>
<|im_start|>assistant
"""

generator = outlines.generate.choice(model, ["Pizza", "Pasta", "Salad", "Dessert"])
answer = generator(prompt)
print(answer)
# Likely answer: Pizza

Pizza


In [17]:
from enum import Enum

class Food(str, Enum):
    pizza = "Pizza"
    pasta = "Pasta"
    salad = "Salad"
    dessert = "Dessert"

generator = outlines.generate.choice(model, Food)
answer = generator(prompt)
print(answer)

Pizza


In [18]:
prompt = "<s>result of 9 + 9 = 18</s><s>result of 1 + 2 = "
answer = outlines.generate.format(model, int)(prompt)
print(answer)
# 3

prompt = "sqrt(2)="
generator = outlines.generate.format(model, float)
answer = generator(prompt, max_tokens=10)
print(answer)
# 1.41421356

KeyboardInterrupt: 

In [19]:
prompt = """
<|im_start|>system You are a helpful assistant.
<|im_end|>

<|im_start|>user
What is an IP address of the Google DNS servers?
<|im_end|>
<|im_start|>assistant
The IP address of a Google DNS server is

"""

generator = outlines.generate.text(model)
unstructured = generator(prompt, max_tokens=30)

generator = outlines.generate.regex(
    model,
    r"((25[0-5]|2[0-4]\d|[01]?\d\d?)\.){3}(25[0-5]|2[0-4]\d|[01]?\d\d?)",
    sampler=outlines.samplers.greedy(),
)
structured = generator(prompt, max_tokens=30)

print(unstructured)
# 8.8.8.8
#
# <|im_end|>

print(structured)
# 8.8.8.8

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


- `8.8.8.8`
- `8.8.4.4`

These addresses are commonly used for DNS resolution. Google
1.8.8.81


In [2]:
import outlines

arithmetic_grammar = """
    ?start: expression

    ?expression: term (("+" | "-") term)*

    ?term: factor (("*" | "/") factor)*

    ?factor: NUMBER
           | "-" factor
           | "(" expression ")"

    %import common.NUMBER
"""

model2 = outlines.models.transformers("WizardLM/WizardMath-7B-V1.1", device="cuda")
generator = outlines.generate.cfg(model2, arithmetic_grammar)
sequence = generator("Alice had 4 apples and Bob ate 2. Write an expression for Alice's apples:")

print(sequence)
# (8-2)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/outlines/fsm/guide.py:110: UserWarning: Outlines' public *community-contributed* CFG structured generation is experimental. Please review https://dottxt-ai.github.io/outlines/latest/reference/generation/cfg#disclaimer
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "E1": {"const": "Anlagevermögen"},
        "E2": {"const": "Immaterielle Vermögensgegenstände"},
        "E3": {"const": "Selbst geschaffene gewerbliche Schutzrechte und ähnliche Rechte und Werte"},
        "Geschäftsjahr": {"type": ["number", "null"]},
        "Vorjahr": {"type": ["number", "null"]}
    },
    "required": ["E1", "E2", "E3", "Geschäftsjahr", "Vorjahr"]
}

* import guidance
* vllm serve background grammar

### to HTML

### to HTML restricted

#### full list

### to CSV

## From md